# 05 – LLM-Topic-Generierung (Ollama / OpenAI)

Dies ist der **letzte Schritt der Topic-Generierungs-Pipeline**. Ein Large Language Model
(standard: `llama3.1:8b` via Ollama, alternativ OpenAI GPT) generiert aus den Evidence-Strings
TREC-konforme Topics.

**Output-Format pro Topic:**
- `TITLE` – Nomen-Phrase, max. 8 Wörter
- `DESC` – 1–2 Sätze zur Suchabsicht
- `NARR` – Bullet-Points zu Relevanz/Nicht-Relevanz

**Voraussetzung:** `04_tfidf.ipynb` wurde ausgeführt. Ollama läuft lokal **oder**
ein OpenAI-API-Key ist in `.env` hinterlegt.

**Ausgabe:** `data/topics_ollama_<variant>.csv` + `runfiles/<run_name>.jsonl`

## 1. Installation

Ollama-Client installieren (nur einmal – danach auskommentiert lassen).

In [1]:
#!pip install ollama


## 2. Imports

Ollama als Standard-Backend. OpenAI-Client ist auskommentiert und kann aktiviert werden.

In [5]:
import pandas as pd
import time
import re
import ollama

## 3. Evidence laden

Liest `data/evidence_topic_generation.csv` aus Notebook 04.
Erwartet: 1.128 Zeilen (376 Sessions × 3 Varianten).

In [6]:
# Cell 3: Evidence laden
evidence_df = pd.read_csv("data/evidence_topic_generation.csv")
print("Evidence-Zeilen:", len(evidence_df))
print(evidence_df["doc_variant"].value_counts())

Evidence-Zeilen: 1128
doc_variant
rel            376
non_rel        376
contrastive    376
Name: count, dtype: int64


## 4. Run-Konfiguration

Alle Parameter für einen einzelnen Run:

| Parameter | Optionen | Beschreibung |
|---|---|---|
| `model_provider` | `"ollama"`, `"openai"` | LLM-Backend |
| `model` | z.B. `"llama3.1:8b"`, `"gpt-4.1-mini"` | Modellname |
| `query_mode` | `"first"`, `"last"`, `"all"` | Welche Query der Session verwendet wird |
| `doc_variant` | `"rel"`, `"non_rel"`, `"contrastive"` | Evidence-Variante aus Notebook 04 |
| `tfidf_top_n` | Integer | Anzahl TF-IDF-Terme im Prompt |
| `run_name` | String | Dateiname des Output-JSONL |

In [7]:
RUN_CONFIG = {
    "model_provider": "ollama",      # "ollama" oder "openai"
    "model": "llama3.1:8b",          # OpenAI z.B. "gpt-4.1-mini"
    "query_mode": "first",           # bei euch Testdaten: meistens "first"
    "doc_variant": "rel",            # "rel", "non_rel", "contrastive"
    "tfidf_top_n": 5,
    "run_name": "ollama_first_rel_tfidf5"
}

## 5. Topic-Generierungs-Funktion

`generate_topic()` sendet den Evidence-String als Prompt an das LLM und parst
die strukturierte Antwort (TITLE / DESC / NARR) via Regex.
Bei fehlerhafter oder leerer Antwort wird bis zu `retries`-mal wiederholt.

Unterstützt Ollama (`ollama.chat`) und OpenAI (`client.chat.completions.create`).

In [8]:
def generate_topic(
    evidence: str,
    model_provider: str = "ollama",
    model: str = "llama3.1:8b",
    retries: int = 2
) -> dict:
    """
    Generiert TITLE, DESC und NARR aus einer Evidence.
    Unterstützt Ollama und OpenAI.
    """

    full_prompt = f"""
You are an expert information retrieval analyst.

Create a search topic with EXACTLY these labeled lines:

TITLE: (<= 8 words)
DESC: (1-2 sentences, clear and specific)
NARR:
- bullet point 1
- bullet point 2
- bullet point 3

Rules:
- Base the topic strictly on the provided evidence.
- Use the search queries as the primary signal.
- Use the TF-IDF terms as supporting evidence only.
- Do NOT introduce unrelated domains.
- Prefer the dominant search intent of the session.
- If the evidence is ambiguous, stay broad and neutral.
- Keep TITLE specific, but not overly narrow.
- TITLE must be a noun phrase.
- Return ONLY the labeled output. No extra text.
- Do NOT use Markdown formatting.
- Do NOT use bold text.

EVIDENCE:
{evidence}
""".strip()

    last_err = None

    for attempt in range(retries + 1):
        try:
            if model_provider == "ollama":
                response = ollama.chat(
                    model=model,
                    messages=[
                        {"role": "user", "content": full_prompt}
                    ],
                    options={
                        "temperature": 0.2,
                        "num_predict": 320
                    }
                )

                out = response["message"]["content"].strip()

            elif model_provider == "openai":
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "user", "content": full_prompt}
                    ],
                    temperature=0.2,
                    max_tokens=320
                )

                out = response.choices[0].message.content.strip()

            else:
                raise ValueError(f"Unknown model_provider: {model_provider}")

            lines = [line.strip() for line in out.splitlines() if line.strip()]

            clean_lines = [
                re.sub(r"^\*+|\*+$", "", line).strip()
                for line in lines
            ]

            title = next(
                (
                    re.sub(r"^TITLE:\s*", "", line, flags=re.IGNORECASE).strip()
                    for line in clean_lines
                    if re.match(r"^TITLE:\s*", line, flags=re.IGNORECASE)
                ),
                ""
            )

            desc = next(
                (
                    re.sub(r"^DESC:\s*", "", line, flags=re.IGNORECASE).strip()
                    for line in clean_lines
                    if re.match(r"^DESC:\s*", line, flags=re.IGNORECASE)
                ),
                ""
            )

            narr_lines = [
                line for line in clean_lines
                if line.startswith("-")
            ]

            narr = "\n".join(narr_lines)

            if not title and attempt < retries:
                time.sleep(0.5)
                continue

            return {
                "title": title,
                "description": desc,
                "narrative": narr,
                "raw_output": out
            }

        except Exception as e:
            last_err = e

            if attempt < retries:
                time.sleep(0.5)
                continue

    return {
        "title": "",
        "description": "",
        "narrative": "",
        "raw_output": f"ERROR: {type(last_err).__name__}: {last_err}"
    }

## 6. Generierungs-Loop

Iteriert über alle Sessions der gewählten `doc_variant` und ruft für jede
`generate_topic()` auf. Alle 200 Zeilen wird ein Checkpoint-CSV gespeichert,
sodass bei Abbruch kein Fortschritt verloren geht.

In [ ]:
# Cell 5: Loop — nur non_rel, für rel oder contrastive erste zeile in dieser zelle anpassen
evidence_df_filtered = evidence_df[evidence_df["doc_variant"] == "non_rel"].copy().reset_index(drop=True)

topic_rows = []

total_rows = len(evidence_df_filtered)

CHECKPOINT_EVERY = 200

for i, row in evidence_df_filtered.iterrows():
    print(
        f"{i+1}/{total_rows} | "
        f"{row['doc_variant']} | "
        f"{row['query_text'][:50]}"
    )

    topic = generate_topic(row["evidence"])

    topic_rows.append({
        "session_id": row["session_id"],
        "doc_variant": row["doc_variant"],
        "query_text": row["query_text"],
        "tfidf_terms": row["tfidf_terms"],
        "evidence": row["evidence"],
        "title": topic["title"],
        "description": topic["description"],
        "narrative": topic["narrative"],
        "raw_output": topic["raw_output"]
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(topic_rows)
        checkpoint_path = f"data/topics_ollama_checkpoint_{i+1}.csv"
        checkpoint_df.to_csv(checkpoint_path, index=False)
        print(f"\nCheckpoint gespeichert: {checkpoint_path}\n")

topics_df = pd.DataFrame(topic_rows)
topics_df.to_csv("data/topics_ollama_non_rel.csv", index=False)

print("\nFertig:", len(topics_df))
print("Final gespeichert: data/topics_ollama_non_rel.csv")

1/376 | non_rel | peer to peer communication
2/376 | non_rel | wattpad reading comprehension
3/376 | non_rel | synthetic biology nasa
4/376 | non_rel | artificial intelligence in education
5/376 | non_rel | procrastination among students
6/376 | non_rel | cannabis
7/376 | non_rel | asphalt engine oil
8/376 | non_rel | Which feeding pattern most strongly predicts eligi
9/376 | non_rel | relationship between financial literacy, financial
10/376 | non_rel | theories about meco waste and coconut shell
11/376 | non_rel | balance in business profit and environmental susta
12/376 | non_rel | vitamin b12
13/376 | non_rel | virtual tour
14/376 | non_rel | the psychological factor affecting student entrepr
15/376 | non_rel | green roof
16/376 | non_rel | political science
17/376 | non_rel | mnemonic device agent based modeling
18/376 | non_rel | How can hands on projects and simulations be combi
19/376 | non_rel | community policing
20/376 | non_rel | effectiveness
21/376 | non_rel | Which featu

## 7. JSONL-Export

Exportiert die Topics im TREC-JSONL-Format: `{qid, query, description, narrative}`.
Diese Dateien können direkt für die Einreichung oder Evaluation verwendet werden.

In [ ]:
# JSONL-Export
import json

output_path = f"runfiles/{RUN_CONFIG['run_name']}.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in topics_df.iterrows():
        entry = {
            "qid": str(row["session_id"]),
            "query": row["query_text"],
            "description": row["description"],
            "narrative": row["narrative"]
        }
        f.write(json.dumps(entry) + "\n")

print(f"Gespeichert: {output_path}")
print(f"Anzahl Topics: {len(topics_df)}")